In [0]:
%run /Repos/dung_nguyen_hoang@mfcgd.com/Utilities/Functions

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
import pandas as pd
#from openpyxl import load_workbook


start_date  = "2023-01-01"
end_date    = "2025-06-30"
inforce_sts = "('1','3','5')"
snapshot_dt = end_date

tpolidm_path = "/mnt/prod/Curated/VN/Master/VN_CURATED_DATAMART_DB/TPOLIDM_MTHEND/"
out_path = "/mnt/lab/vn/project/VN_CLAIMS/CLAIMS/"

### 1_Metrics to load
<i>po_num<br>
clm_id<br>
clm_eff_dt<br>
clm_aprv_amt/pmt_dt<br>
clm_benefit_type<br>
product_types<br>
rider_types<br>
APE<br>
agt_segments<br>
tNPS</i>

In [0]:
# Load latest Agency Structure from MIS
loc_path = f"/mnt/lab/vn/project/scratch/adhoc/loc_to_psm_mapping_202505.xlsx"
sheet_name = "Structure"
cell_pos = "A1"
loc_to_psm = spark.read.format(
    "com.crealytics.spark.excel"
).option(
    "dataAddress", f"{sheet_name}!{cell_pos}"
).option(
    "header", "true"
).option(
    "treatEmptyValuesAsNulls", "true"
).option(
    "inferSchema", "true"
).option(
    "addColorColumns", "False"
).load(
    loc_path
).select(
    "loc_cd", "psm_name_8", "psm_name_9"
)

# Derive Agent status and Group
mpro_title = spark.sql(f"""
SELECT  agt.AGT_CODE AS agt_code,
        CASE WHEN stat_cd = '01' THEN 'INFORCE'
             WHEN stat_cd = '99' THEN 'TERMINATED'
        END AS agt_status,
        CASE WHEN trmn_dt IS NOT NULL THEN '4.Terminated'
             WHEN trmn_dt IS NULL THEN
                 CASE
                     WHEN comp_prvd_num IN ('01','02','08','96') THEN '1.Inforce'
                     WHEN comp_prvd_num = '04' THEN '2.CSC'
                     WHEN comp_prvd_num IN ('97','98') THEN '3.SM'
                     ELSE '5.Not-Agency'
                 END
        END AS agt_rltnshp,
        CASE WHEN trmn_dt IS NOT NULL OR comp_prvd_num IN ('04','97','98') THEN 1 ELSE 0 
        END AS unassigned_ind,
        COALESCE(
            CASE WHEN agt.MPRO_TITLE IS NOT NULL THEN
                CASE WHEN MDRT_DESC IN ('TOT', 'COT') THEN 'MDRT'
                     ELSE MDRT_DESC
                END
                ELSE 'Normal'
            END,
            MPRO_TITLE
        ) AS mpro_title,
        agt.LOC_CODE AS loc_cd,
        agt.image_date
FROM    vn_published_casm_ams_snapshot_db.tams_agents agt
WHERE   1=1
    AND agt.image_date = '{snapshot_dt}'
""")


par_df = spark.read.format("parquet").load(
    f"/mnt/lab/vn/project/cpm/datamarts/TPARDM_MTHEND/"
).select(
    "agt_cd", "last_9m_pol", "last_3m_pol", "last_mth_pol", "image_date"
).withColumnRenamed(
    "agt_cd", "agt_code"
)

mpro_title = mpro_title.join(
    par_df, 
    on=["agt_code", "image_date"],
    how="left"
).join(
    loc_to_psm,
    "loc_cd",
    "left"
)

agt_df = mpro_title.withColumn(
    "agt_actvness",
    F.when(F.col("last_mth_pol") > 0, "1m active")
    .when(F.col("last_3m_pol") > 0, "3m active")
    .when(F.col("last_9m_pol") > 0, "9m active")
    .otherwise("SA")
).withColumn(
    "agt_segment",
    F.when(F.col("agt_actvness") != "SA",
        F.when(F.col("mpro_title") != "Normal", F.col("mpro_title"))
        .otherwise(F.col("agt_actvness"))
    ).otherwise(
        F.when(F.col("agt_rltnshp").isin("2.CSC","3.SM","4.Terminated"), "UCM")
        .otherwise("SA")
    )
).drop(
    "last_3m_pol", "last_9m_pol", "last_mth_pol", "agt_status", "agt_rltnshp", "agt_actvness"
)

In [0]:
po_pol_list = spark.sql(f"""
WITH frst_pol AS (
    SELECT  cpl.CLI_NUM,
            MIN(pol.POL_ISS_DT) AS FRST_ISS_DT,
            cpl.image_date
    FROM    vn_published_casm_cas_snapshot_db.tpolicys pol
        INNER JOIN
            vn_published_casm_cas_snapshot_db.tclient_policy_links cpl
            ON pol.POL_NUM = cpl.POL_NUM AND pol.image_date = cpl.image_date AND
               cpl.LINK_TYP = 'O' AND cpl.REC_STATUS = 'A' 
    WHERE   cpl.image_date = '{snapshot_dt}'
        AND POL_STAT_CD NOT IN ('A','N','R','X')
    GROUP BY
            cpl.CLI_NUM, cpl.image_date
), ape_holding AS (
    SELECT  image_date, POL_NUM,
            CAST(SUM(CASE WHEN CVG_TYP = 'B' THEN CVG_PREM END) AS INT) AS base_ape,
            CAST(SUM(CASE WHEN CVG_TYP = 'R' THEN CVG_PREM END) AS INT) AS rid_ape,
            CAST(SUM(CVG_PREM) AS INT) AS tot_ape
    FROM    vn_published_casm_cas_snapshot_db.tcoverages
    WHERE   image_date = '{snapshot_dt}'
        AND CVG_TYP IN ('B','R')
    GROUP BY
            image_date, POL_NUM
)   
SELECT  cpl.CLI_NUM     AS po_num,
        pol.POL_NUM     AS pol_num,
        TO_DATE(pol.POL_ISS_DT)
                        AS pol_iss_dt,
        TO_DATE(fpol.FRST_ISS_DT)
                        AS frst_iss_dt,
        pol.AGT_CODE    AS agt_code,
        CASE WHEN agt.TRMN_DT IS NOT NULL OR
                  agt.COMP_PRVD_NUM IN ('04','97','98') THEN 1
                  ELSE 0
        END AS unassigned_ind,
        ape.base_ape,
        ape.rid_ape,
        ape.tot_ape,
        ROW_NUMBER() OVER (PARTITION BY cpl.CLI_NUM ORDER BY pol.POL_EFF_DT DESC) AS rn,
        pol.image_date
FROM    vn_published_casm_cas_snapshot_db.tpolicys pol
    INNER JOIN
        vn_published_casm_cas_snapshot_db.tclient_policy_links cpl
        ON pol.POL_NUM = cpl.POL_NUM AND pol.image_date = cpl.image_date AND
           cpl.LINK_TYP = 'O' AND cpl.REC_STATUS = 'A' 
    INNER JOIN
        frst_pol fpol
        ON cpl.CLI_NUM = fpol.CLI_NUM AND cpl.image_date = fpol.image_date
    INNER JOIN
        ape_holding ape
        ON pol.POL_NUM = ape.POL_NUM AND pol.image_date = ape.image_date
    INNER JOIN
        vn_published_casm_ams_snapshot_db.tams_agents agt
        ON pol.AGT_CODE = agt.AGT_CODE AND pol.image_date = agt.image_date
WHERE   pol.image_date = '{snapshot_dt}'
    AND pol.POL_STAT_CD IN {inforce_sts}
""")

#check_dup(po_pol_list, "pol_num")
po_pol_list.createOrReplaceTempView("nb")

po_list = po_pol_list.groupBy(
    "image_date", "po_num"
).agg(
    F.min("frst_iss_dt").alias("frst_iss_dt"),
    F.sum("base_ape").alias("base_ape"),
    F.sum("rid_ape").alias("rider_ape"),
    F.sum("tot_ape").alias("total_ape"),
    F.sum("unassigned_ind").alias("unassigned_ind")
)

po_list = po_list.withColumn(
    "total_ape_cat",
    F.when(F.col("total_ape") <= 10000, "a. <=10M")
     .when(F.col("total_ape") <= 15000, "b. 10 - 15M")
     .when(F.col("total_ape") <= 20000, "c. 15 - 20M")
     .when(F.col("total_ape") <= 25000, "d. 20 - 25M")
     .when(F.col("total_ape") <= 30000, "e. 25 - 30M")
     .when(F.col("total_ape") <= 50000, "f. 30 - 50M")
     .when(F.col("total_ape") > 50000, "g. >50M")
     .otherwise("h. Missing")
).withColumn(
    "base_ape_cat",
    F.when(F.col("base_ape") <= 10000, "a. <=10M")
     .when(F.col("base_ape") <= 15000, "b. 10 - 15M")
     .when(F.col("base_ape") <= 20000, "c. 15 - 20M")
     .when(F.col("base_ape") <= 25000, "d. 20 - 25M")
     .when(F.col("base_ape") <= 30000, "e. 25 - 30M")
     .when(F.col("base_ape") <= 50000, "f. 30 - 50M")
     .when(F.col("base_ape") > 50000, "g. >50M")
     .otherwise("h. Missing")
).withColumn(
    "rider_ape_cat",
    F.when(F.col("rider_ape") <= 5000, "a. <=5M")
     .when(F.col("rider_ape") <= 7000, "b. 5 - 7M")
     .when(F.col("rider_ape") <= 10000, "c. 7 - 10M")
     .when(F.col("rider_ape") <= 15000, "d. 10 - 15M")
     .when(F.col("rider_ape") <= 20000, "e. 15 - 20M")
     .when(F.col("rider_ape") <= 30000, "f. 20 - 30M")
     .when(F.col("rider_ape") > 30000, "g. >30M")
     .otherwise("h. Missing")
).withColumn(
    "full_ucm_ind",
    F.when(F.col("unassigned_ind") > 0, 1)
    .otherwise(0)
)

check_dup(po_list, "po_num")

po_last_pol = po_pol_list.filter(
    F.col("rn") == 1
).select(
    "image_date", "po_num", "agt_code"
)

po_last_pol = po_last_pol.join(
    agt_df,
    on=["image_date", "agt_code"]
)

check_dup(po_last_pol, "po_num")

In [0]:
claim_df = spark.sql(f"""
-- retrieve claim payment dates
WITH results_dedup AS (
  SELECT DISTINCT
         POL_NUM,
         CLI_NUM,
         RULE_ID,
         err_en_msg, TYPE, ASSIGN_TEAM, trxn_dt, rerun_seq
         ,DENSE_RANK() OVER (PARTITION BY POL_NUM ORDER BY rerun_seq DESC) AS rank
  FROM   vn_published_cas_db.tpolicy_results
  QUALIFY rank = 1
), order_results AS (
  SELECT POL_NUM,
         CLI_NUM,
         RULE_ID,
         err_en_msg,
         CASE 
           WHEN TYPE = 'C' THEN 'C-Client'
           WHEN TYPE = 'P' THEN 'P-Policy'
           WHEN TYPE = 'A' THEN 'A-Agent'
           ELSE TYPE
         END AS TYPE,
         ASSIGN_TEAM,
         ROW_NUMBER() OVER(PARTITION BY POL_NUM ORDER BY POL_NUM) AS rn
  FROM   results_dedup
),
sorted_remarks AS (
  SELECT POL_NUM, CLI_NUM, RULE_ID, TYPE, ASSIGN_TEAM, err_en_msg
  FROM order_results
),
clm_his AS (
SELECT POL_NUM, CLI_NUM,
       CONCAT_WS('/', COLLECT_SET(TYPE)) AS TYPES,
       CONCAT_WS('>', COLLECT_LIST(err_en_msg)) AS ADMIN_REMARKS
FROM sorted_remarks
WHERE   TYPE='C-Client'
  AND   err_en_msg like '%Client has claim history%'
GROUP BY POL_NUM, CLI_NUM
), 
clm_type as (
select  cast(fld_valu as int) AS clm_code
        , fld_valu_desc AS benefit
        ,case when cast(fld_valu as int) in (3,7,9,36,38) then 'Medicash'
          when cast(fld_valu as int) in (11,27,28,29,31,40,41,42,45,50,51) then 'Healthcare'
          when cast(fld_valu as int) in (5,21,23,24,25,30,33,34,35) then 'Major Diseases'
          when cast(fld_valu as int) in (4,20,37) then 'Major Death'
          else 'Other Major Benefits'
     end AS clm_benefit_type
from    vn_published_cas_db.tfield_values
where   fld_nm='CLM_CODE'
), 
t_dnr_sub_a as (
	select
		payee
		,max(pmt_dt) pmt_dt
	from
		vn_published_cas_db.tdnr_details
	where
		dnr_stat_cd <> 'X'
		and pmt_stat_cd not in ('N','W')
		and payo_reas in ('912','913','914','915','916','917','918','919','920','921','927','928','929','930')							
	group by
		payee
),
t_dnr_sub_b as (
	select
		payee
		,pmt_dt
		,max(payo_type) payo_type
	from
		vn_published_cas_db.tdnr_details
	where
		dnr_stat_cd <> 'X'
		and pmt_stat_cd not in ('N','W')
		and payo_reas in ('912','913','914','915','916','917','918','919','920','921','927','928','929','930')							
	group by
		payee
		,pmt_dt
),
t_dnr AS (
select
	a.payee
	,case
		when c.direct_billing = 'Y' then c.clm_recv_dt
		when c.payo_typ = 'ANM' then c.clm_aprov_dt
		when c.payo_typ is null and c.case_owner in ('INSMARTHCM','INSMARTHN') then date_add(c.clm_aprov_dt,3)
		when c.payo_typ in ('TF','PO') then date_add(c.clm_aprov_dt,3)
		when a.pmt_dt is null then c.clm_aprov_dt
		else a.pmt_dt
	end pmt_dt
	,b.payo_type
from	
	t_dnr_sub_a a
	inner join t_dnr_sub_b b on	(a.payee = b.payee and a.pmt_dt = b.pmt_dt)
    inner join vn_published_cas_db.tclaim_details c on a.payee = c.clm_id
),
all_claims as (
SELECT	CLM_ID, POL_NUM, CLI_NUM, CLM_CODE, CLM_EFF_DT, CLM_APP_DT, CLM_PRVD_AMT, CLM_APROV_AMT, CLM_STAT_CODE
FROM   	vn_published_cas_db.tclaim_details
UNION
SELECT	CLM_ID, POL_NUM, con.CLI_CNSLDT_NUM AS CLI_NUM, CLM_CODE, CLM_EFF_DT, CLM_APP_DT, CLM_PRVD_AMT, CLM_APROV_AMT, CLM_STAT_CODE
FROM   	vn_published_cas_db.tclaim_details clm
  INNER JOIN
        vn_published_cas_db.tclient_consolidations con ON clm.CLI_NUM = con.CLI_NUM
)
SELECT DISTINCT
    nb.po_num
    ,clm.CLM_ID
    ,clm.POL_NUM
    ,clm.CLI_NUM
    ,TO_DATE(clm.CLM_EFF_DT) AS CLM_DT
    ,YEAR(clm.CLM_EFF_DT) AS CLM_YR
	  ,SUBSTR(TO_DATE(clm.CLM_EFF_DT),1,7) AS CLM_MTH
    ,SUBSTR(TO_DATE(t_dnr.pmt_dt),1,7) AS PMT_MTH
    ,CAST(NVL(clm.CLM_PRVD_AMT,0) AS INT) AS CLM_AMT
    ,CAST(CASE WHEN clm.CLM_STAT_CODE = 'A' THEN clm.CLM_APROV_AMT ELSE 0 END AS INT) AS CLM_APRV_AMT
    ,typ.clm_benefit_type
    ,CASE WHEN clm.CLM_EFF_DT > nb.FRST_ISS_DT THEN 0 ELSE 1 END AS VALID_HIS_CLM_IND
    ,CASE WHEN clm.CLM_STAT_CODE = 'A' THEN 1 ELSE 0 END AS CLM_APRV_IND
    ,nb.image_date AS image_date
FROM  parquet.`{tpolidm_path}` nb 
  INNER JOIN 
      all_claims clm 
      ON nb.POL_NUM = clm.POL_NUM
  LEFT JOIN
      clm_his his 
      ON clm.cli_num = his.cli_num 
  LEFT JOIN
      t_dnr 
      ON clm.clm_id=t_dnr.payee 
  LEFT JOIN
      clm_type typ 
      ON clm.clm_code=typ.clm_code
WHERE   1=1
  AND   nb.image_date = '{snapshot_dt}'
  AND   TO_DATE(clm.CLM_EFF_DT) BETWEEN '{start_date}' AND '{end_date}'
  AND   clm.CLM_STAT_CODE IN ('A','D')
""")

# List of benefit types and their short labels
benefit_types = {
    'Medicash': 'MC',
    'Healthcare': 'MR',
    'Major Diseases': 'DISEASES',
    'Major Death': 'DEATH',
    'Other Major Benefits': 'OTHER'
}

# Base aggregations
aggregations = [
    F.count("CLM_ID").alias("total_submitted_claims"),
    F.sum("CLM_APRV_IND").alias("total_approved_claims"),
    F.sum("CLM_AMT").alias("total_submitted_claim_amount"),
    F.sum("CLM_APRV_AMT").alias("total_approved_claim_amount"),
    F.concat_ws("/", F.collect_set("clm_benefit_type")).alias("claim_benefits"),
    F.countDistinct("CLM_YR").alias("claim_years"),
    F.concat_ws("/", F.collect_set("CLM_MTH")).alias("claim_months"),
    F.max("VALID_HIS_CLM_IND").alias("VALID_HIS_CLM_IND")
]

# Dynamically add aggregations for each benefit type
for benefit, label in benefit_types.items():
    aggregations.extend(
      [
        F.count(F.when(F.col("clm_benefit_type") == benefit, F.col("pol_num")))
        .alias(f"{label}_claims"),

        F.count(F.when((F.col("clm_benefit_type") == benefit) & 
                       (F.col("CLM_APRV_IND") == 1), F.col("pol_num")))
        .alias(f"{label}_approved_claims"),

        F.sum(F.when(F.col("clm_benefit_type") == benefit, F.col("clm_amt"))
              .otherwise(0))
        .alias(f"{label}_claim_amount"),

        F.sum(F.when((F.col("clm_benefit_type") == benefit) & 
                     (F.col("CLM_APRV_IND") == 1), F.col("clm_amt"))
              .otherwise(0))
        .alias(f"{label}_approved_claim_amount")
    ]
)

# Apply the aggregation
claim_sum = claim_df.groupBy(
  "image_date", "po_num"
).agg(
  *aggregations
)

# Identify customers with multiple claim types
claim_columns = [F.col(f"{label}_claims") > 0 for label in benefit_types.values()]
multiple_claim_condition = sum(F.when(cond, 1).otherwise(0) for cond in claim_columns) > 1

# Add the multiple_claim_ind column
claim_sum = claim_sum.withColumn(
  "multiple_claim_ind", 
  F.when(multiple_claim_condition, "Y").otherwise("N")
)

# print(claim_sum.count())
# check_dup(claim_sum, "po_num")

In [0]:
# Finding desire base (as of latest date)
product_portfolio = spark.sql(f"""
with prod_list as (
select  PLAN_CODE, INS_TYP
        , case when fld.FLD_VALU_DESC='Investment' then
                case when pln.PLAN_CODE like 'RUV%' then 'Investment-ILP'
                    when pln.PLAN_CODE like 'UL%' then 'Investment-UL'
                    else 'Investment-Others' end
               when fld.FLD_VALU_DESC='Whole Life' then
                case when pln.PLAN_CODE not like '%ED99' then 'Endowment'
                    else 'Whole Life' end
               when pln.PLAN_CODE in ('FDB01','BIC01','BIC02','BIC03','BIC04') then 'Digital'
               when pln.PLAN_CODE in ('BHC9I','CA360','CI360','CN360','CX360','PN001') then 'Micro-Activator'
              else fld.FLD_VALU_DESC end
         as PROD_TYPE
from    vn_published_cas_db.tplans pln inner join
        vn_published_cas_db.tfield_values fld on pln.INS_TYP = fld.FLD_VALU and fld.FLD_NM='INS_TYP'
where   1=1
  and   pln.CVG_TYP = 'B'
), inf_cus as (
select    -- holding RUV
          distinct cpl.CLI_NUM as PO_NUM
          , cpl.image_date
          , collect_set(pln.PROD_TYPE) as LIST_OF_PRODUCT_TYPES
          , collect_set(rpl.PROD_TYP) as LIST_OF_RIDER_TYPES
from      vn_published_casm_cas_snapshot_db.tpolicys pol 
    inner join
          vn_published_casm_cas_snapshot_db.tclient_policy_links cpl on pol.POL_NUM = cpl.POL_NUM and cpl.LINK_TYP='O' and cpl.REC_STATUS='A' and pol.image_date = cpl.image_date
    inner join
          vn_published_casm_ams_snapshot_db.tams_agents agt on pol.AGT_CODE = agt.AGT_CODE and pol.image_date = agt.image_date
    left join
          vn_published_casm_cas_snapshot_db.tcoverages rid on pol.POL_NUM = rid.POL_NUM and rid.CVG_TYP='R' 
          --and rid.CVG_STAT_CD in ('1','2','3','5','7','9') 
          and pol.image_date = rid.image_date
    left join 
          prod_list pln on pol.PLAN_CODE_BASE = pln.PLAN_CODE 
    left join
          vn_published_cas_db.tplans rpl on rid.PLAN_CODE = rpl.PLAN_CODE
where     1=1
    and   pol.image_date = '{snapshot_dt}'
    and     pol.POL_STAT_CD in {inforce_sts}
group by  cpl.CLI_NUM, cpl.image_date
)
select    a.*
from      inf_cus a
""")

# Step 1: Create indicator columns for product types
result = product_portfolio.withColumn(
    "ILP_IND", F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Investment-ILP"), "Y").otherwise("N")
).withColumn(
    "UL_IND", F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Investment-UL"), "Y").otherwise("N")
).withColumn(
    "ENDOWMENT_IND", F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Endowment"), "Y").otherwise("N")
).withColumn(
    "WHOLE_IND", F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Whole Life"), "Y").otherwise("N")
).withColumn(
    "TERM_IND", F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Term Life"), "Y").otherwise("N")
).withColumn(
    "DIGITAL_IND", F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Digital"), "Y").otherwise("N")
).withColumn(
    "MICRO_IND", F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Micro-Activator"), "Y").otherwise("N")
).withColumn(
    "OTH_IND", F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Accident"), "Y").otherwise("N")
)

# Step 2: Explode LIST_OF_RIDER_TYPES into dynamic columns
unique_rider_types = result.select(F.explode(F.col("LIST_OF_RIDER_TYPES")).alias("rider")).distinct().collect()
rider_type_columns = [rider.rider for rider in unique_rider_types]

for rider_type in rider_type_columns:
    result = result.withColumn(
        f"{rider_type}_IND",
        F.when(F.array_contains(F.col("LIST_OF_RIDER_TYPES"), rider_type), "Y").otherwise("N")
    )

portfolio = result.drop('LIST_OF_PRODUCT_TYPES','LIST_OF_RIDER_TYPES')

In [0]:
# Add `rider_ind` and `product_type` flag
new_cols = ({
    'rider_ind': F.greatest(F.col('cancer_rider_ind'),
                            F.col('add_rider_ind'),
                            F.col('tpd_rider_ind'),
                            F.col('tp_rider_ind'),
                            F.col('mc_rider_ind'),
                            F.col('ci_rider_ind'),
                            F.col('hc_rider_ind')),

    'product_type': F.when((F.col('ilp_ind') == 'Y') &
                           (F.greatest(F.col('ul_ind'),
                                     F.col('endowment_ind'),
                                     F.col('whole_ind'),
                                     F.col('term_ind'),
                                     F.col('digital_ind'),
                                     F.col('micro_ind'),
                                     F.col('oth_ind')) == 'N'),
                           'ILP only')
                    .when((F.col('ul_ind') == 'Y') &
                           (F.greatest(F.col('ilp_ind'),
                                     F.col('endowment_ind'),
                                       F.col('whole_ind'),
                                       F.col('term_ind'),
                                       F.col('digital_ind'),
                                       F.col('micro_ind'),
                                       F.col('oth_ind')) == 'N'),
                           'UL only')
                    .when((F.col('endowment_ind') == 'Y') &
                          (F.greatest(F.col('ilp_ind'),
                                     F.col('ul_ind'),
                                      F.col('whole_ind'),
                                      F.col('term_ind'),
                                      F.col('digital_ind'),
                                      F.col('micro_ind'),
                                      F.col('oth_ind')) == 'N'),
                          'Endowment only')
                    .when((F.col('whole_ind') == 'Y') &
                          (F.greatest(F.col('ilp_ind'),
                                     F.col('ul_ind'),
                                      F.col('endowment_ind'),
                                      F.col('term_ind'),
                                      F.col('digital_ind'),
                                      F.col('micro_ind'),
                                      F.col('oth_ind')) == 'N'),
                          'Whole Life only')
                    .when((F.col('term_ind') == 'Y') &
                          (F.greatest(F.col('ilp_ind'),
                                     F.col('ul_ind'),
                                      F.col('endowment_ind'),
                                      F.col('whole_ind'),
                                      F.col('digital_ind'),
                                      F.col('micro_ind'),
                                      F.col('oth_ind')) == 'N'),
                          'Term Life only')
                    .when((F.col('digital_ind') == 'Y') &
                          (F.greatest(F.col('ilp_ind'),
                                     F.col('ul_ind'),
                                      F.col('endowment_ind'),
                                      F.col('whole_ind'),
                                      F.col('term_ind'),
                                      F.col('micro_ind'),
                                      F.col('oth_ind')) == 'N'),
                          'Digital only')
                    .when((F.col('micro_ind') == 'Y') &
                          (F.greatest(F.col('ilp_ind'),
                                     F.col('ul_ind'),
                                      F.col('endowment_ind'),
                                      F.col('whole_ind'),
                                      F.col('term_ind'),
                                      F.col('digital_ind'),
                                      F.col('oth_ind')) == 'N'),
                          'Micro only')
                    .when((F.col('oth_ind') == 'Y') &
                          (F.greatest(F.col('ilp_ind'),
                                      F.col('ul_ind'),
                                      F.col('endowment_ind'),
                                      F.col('whole_ind'),
                                      F.col('term_ind'),
                                      F.col('digital_ind'),
                                      F.col('micro_ind')) == 'N'),
                          'Accident only')
                    .when(F.greatest(F.col('ilp_ind'),
                                    F.col('ul_ind'),
                                    F.col('endowment_ind'),
                                    F.col('whole_ind'),
                                    F.col('term_ind'),
                                    F.col('digital_ind'),
                                    F.col('micro_ind'),
                                    F.col('oth_ind')) == 'Y', 'Multiple products'),

    'rider_type': F.when((F.col('mc_rider_ind') == 'Y') &
                         (F.col('hc_rider_ind') != 'Y') &
                         (F.greatest(F.col('cancer_rider_ind'),
                                     F.col('add_rider_ind'),
                                     F.col('tpd_rider_ind'),
                                     F.col('tp_rider_ind'),
                                     F.col('ci_rider_ind')) != 'Y'),
                         'MC Only')
                  .when((F.col('hc_rider_ind') == 'Y') &
                        (F.col('mc_rider_ind') != 'Y') &
                        (F.greatest(F.col('cancer_rider_ind'),
                                    F.col('add_rider_ind'),
                                    F.col('tpd_rider_ind'),
                                    F.col('tp_rider_ind'),
                                    F.col('ci_rider_ind')) != 'Y'),
                        'HC Only')
                  .when((F.col('mc_rider_ind') == 'Y') &
                        (F.col('hc_rider_ind') == 'Y') &
                        (F.greatest(F.col('cancer_rider_ind'),
                                    F.col('add_rider_ind'),
                                    F.col('tpd_rider_ind'),
                                    F.col('tp_rider_ind'),
                                    F.col('ci_rider_ind')) != 'Y'),
                        'MC & HC')
                  .when((F.col('hc_rider_ind') == 'Y') &
                        (F.greatest(F.col('mc_rider_ind'),
                                    F.col('cancer_rider_ind'),
                                    F.col('add_rider_ind'),
                                    F.col('tpd_rider_ind'),
                                    F.col('tp_rider_ind'),
                                    F.col('ci_rider_ind')) == 'Y'),
                        'Multiple with HC')
                  .when((F.col('mc_rider_ind') == 'Y') &
                        (F.greatest(F.col('hc_rider_ind'),
                                    F.col('cancer_rider_ind'),
                                    F.col('add_rider_ind'),
                                    F.col('tpd_rider_ind'),
                                    F.col('tp_rider_ind'),
                                    F.col('ci_rider_ind')) == 'Y'),
                        'Multiple with MC')
                  .otherwise('Others')
})

portfolio = portfolio.withColumns(new_cols)
portfolio = portfolio.toDF(*[col.lower() for col in portfolio.columns])

In [0]:
loc_path = "/mnt/lab/vn/project/VN_CLAIMS/CLAIMS/csv/claim_tnps_customer_list.xlsx"
sheet_names = ["pol_list", "cli_list"]
dfs = {}

for sheet in sheet_names:
    df = spark.read.format("com.crealytics.spark.excel") \
        .option("dataAddress", f"{sheet}!A1") \
        .option("header", "true") \
        .option("treatEmptyValuesAsNulls", "true") \
        .option("inferSchema", "true") \
        .option("addColorColumns", "False") \
        .load(loc_path) \
        .withColumn(
            "tnps_cat",
            F.when(F.col("tnps") <= 6, "Detractor")
            .when(F.col("tnps") <= 8, "Indifference")
            .otherwise("Promoter")) \
        .withColumn(
        "image_date", 
        F.last_day(F.col("trxn_dt")))
    dfs[sheet] = df

# print(dfs["pol_list"].count())
# print(dfs["cli_list"].count())

In [0]:
all_claims = spark.sql(f"""
WITH raw_claims AS (
SELECT	CLM_ID, POL_NUM, CLI_NUM, CLM_CODE, CLM_PRVD_AMT, CLM_APROV_AMT, CLM_STAT_CODE
FROM   	vn_published_cas_db.tclaim_details
UNION
SELECT	CLM_ID, POL_NUM, con.CLI_CNSLDT_NUM AS CLI_NUM, CLM_CODE, CLM_PRVD_AMT, CLM_APROV_AMT, CLM_STAT_CODE
FROM   	vn_published_cas_db.tclaim_details clm
  INNER JOIN
        vn_published_cas_db.tclient_consolidations con ON clm.CLI_NUM = con.CLI_NUM
), 
clm_type as (
select  cast(fld_valu as int) AS clm_code
        , fld_valu_desc AS benefit
        ,case when cast(fld_valu as int) in (3,7,9,36,38) then 'Medicash'
          when cast(fld_valu as int) in (11,27,28,29,31,40,41,42,45,50,51) then 'Healthcare'
          when cast(fld_valu as int) in (5,21,23,24,25,30,33,34,35) then 'Major Diseases'
          when cast(fld_valu as int) in (4,20,37) then 'Major Death'
          else 'Other Major Benefits'
     end AS clm_benefit_type
from    vn_published_cas_db.tfield_values
where   fld_nm='CLM_CODE'
)
SELECT  po.PO_NUM, clm.*,
        typ.clm_benefit_type
FROM    raw_claims clm
    INNER JOIN
        parquet.`{tpolidm_path}` po ON clm.POL_NUM = po.POL_NUM
    LEFT JOIN
        clm_type typ ON clm.CLM_CODE = typ.CLM_CODE
""")

clm_cols_1 = ["po_num", "POL_NUM", "CLM_ID", "clm_benefit_type", "CLM_PRVD_AMT", "CLM_APROV_AMT", "CLM_STAT_CODE"]
clm_pol_dedup = all_claims.select(*clm_cols_1).distinct()

# Derive Claim tNPS from policy list
claim_tnps_p1 = dfs["pol_list"].alias("a").join(
    clm_pol_dedup.alias("b"),
    (F.col("a.pol_num") == F.col("b.pol_num")) | 
    (F.col("a.clm_id") == F.col("b.clm_id")),
    how="left"
).withColumn(
    "tnps_clm_benefit_type",
    F.when(
        F.col("a.clm_id").isNotNull() &
        (F.col("a.clm_id") == F.col("b.clm_id")),
        F.col("b.clm_benefit_type"))
    .otherwise(None)
).withColumn(
    "tnps_claim_status",
    F.when(
        F.col("a.clm_id").isNotNull() &
        (F.col("a.clm_id") == F.col("b.clm_id")),
        F.col("CLM_STAT_CODE"))
    .otherwise(None)
).withColumn(
    "tnps_claim_amount",
    F.when(
        F.col("a.clm_id").isNotNull() &
        (F.col("a.clm_id") == F.col("b.clm_id")),
        F.col("CLM_PRVD_AMT"))
    .otherwise(0)
).withColumn(
    "tnps_approved_claim_amount",
    F.when(
        F.col("a.clm_id").isNotNull() &
        (F.col("a.clm_id") == F.col("b.clm_id")),
        F.col("CLM_APROV_AMT"))
    .otherwise(0)
).distinct()

clm_cols_2 = ["po_num", "CLI_NUM", "CLM_ID", "clm_benefit_type", "CLM_PRVD_AMT", "CLM_APROV_AMT", "CLM_STAT_CODE"]
clm_cli_dedup = all_claims.select(*clm_cols_2).distinct()

# Derive Claim tNPS from client list
claim_tnps_p2 = dfs["cli_list"].alias("a").join(
    clm_cli_dedup.alias("b"),
    (F.col("a.cli_num") == F.col("b.cli_num")) | 
    (F.col("a.clm_id") == F.col("b.clm_id")) |
    (F.col("a.cli_num") == F.col("b.po_num")),
    how="left"
).withColumn(
    "tnps_clm_benefit_type",
    F.when(
        F.col("a.clm_id").isNotNull() &
        (F.col("a.clm_id") == F.col("b.clm_id")),
        F.col("b.clm_benefit_type"))
    .otherwise(None)
).withColumn(
    "tnps_claim_status",
    F.when(
        F.col("a.clm_id").isNotNull() &
        (F.col("a.clm_id") == F.col("b.clm_id")),
        F.col("CLM_STAT_CODE"))
    .otherwise(None)
).withColumn(
    "tnps_claim_amount",
    F.when(
        F.col("a.clm_id").isNotNull() &
        (F.col("a.clm_id") == F.col("b.clm_id")),
        F.col("CLM_PRVD_AMT"))
    .otherwise(0)
).withColumn(
    "tnps_approved_claim_amount",
    F.when(
        F.col("a.clm_id").isNotNull() &
        (F.col("a.clm_id") == F.col("b.clm_id")),
        F.col("CLM_APROV_AMT"))
    .otherwise(0)
).distinct()

claim_tnps_merged = claim_tnps_p1.select(
    "image_date", "po_num", "tnps", "tnps_cat", "tnps_clm_benefit_type",
    "tnps_claim_status", "tnps_claim_amount", "tnps_approved_claim_amount"
).union(claim_tnps_p2.select(
    "image_date", "po_num", "tnps", "tnps_cat", "tnps_clm_benefit_type",
    "tnps_claim_status", "tnps_claim_amount", "tnps_approved_claim_amount"
    )
).withColumn(
    "tnps_claim_status",
    F.when(F.col("tnps_claim_status") == "A", "Approved")
    .when(F.col("tnps_claim_status").isin(["D","I"]), "Declined/Closed")
    .when(F.col("tnps_claim_status") == "P", "Pending")
    .otherwise("")
).withColumnRenamed(
    "tnps", "score"
).withColumnRenamed(
    "tnps_cat", "tnps"
).distinct()

# Create a window specification to order by image_date
window_spec = Window.partitionBy("po_num").orderBy(F.col("image_date").desc())

# Add a row number to each row within the po_num partition
df_with_row_num = claim_tnps_merged.withColumn("row_num", F.row_number().over(window_spec))

# Cache the DataFrame with row numbers
df_with_row_num.cache()

# Filter to get the latest record for each po_num
claim_tnps_dedup = df_with_row_num.filter(F.col("row_num") == 1).drop("row_num")

# Filter to get the duplicated records and select the lowest row_num for each po_num
dup_df = df_with_row_num.filter(
    F.col("row_num") > 1
).withColumn(
    "row_num_dup", 
    F.row_number().over(Window.partitionBy("po_num").orderBy(F.col("row_num")))
).filter(
    F.col("row_num_dup") == 1
).select(
    F.col("po_num"),
    F.col("score").alias("score_before"),
    F.col("tnps").alias("tnps_before"),
    #F.col("image_date").alias("image_date_before")
)

# Join the duplicated records back to the unique records
claim_tnps = claim_tnps_dedup.join(
    dup_df, 
    on="po_num", 
    how="left"
)

# check_dup(claim_tnps, "po_num")

### 2_Merge all metrics to Claim data

In [0]:
result = claim_sum.join(
    po_list,
    on=["image_date", "po_num"],
    how="inner"
).join(
    po_last_pol,
    on=["image_date", "po_num"],
    how="inner"
).join(
    portfolio,
    on=["image_date", "po_num"],
    how="left"
).join(
    claim_tnps.select(
        "po_num", "score", "tnps", "score_before", "tnps_before", "tnps_clm_benefit_type",
        "tnps_claim_status", "tnps_claim_amount", "tnps_approved_claim_amount"),
    on="po_num",
    how="left"
).drop(
    "ilp_ind", "ul_ind", "endowment_ind", "whole_ind",
    "term_ind", "digital_ind", "micro_ind", "oth_ind", "unassigned_ind",
    "mc_rider_ind", "hc_rider_ind", "cancer_rider_ind", "add_rider_ind",
    "tpd_rider_ind", "tp_rider_ind", "ci_rider_ind", "rider_ind",
    "base_ape", "rider_ape", "total_ape", "loc_cd"
).fillna(
    {
        "psm_name_8": "Open",
        "tnps_claim_amount": 0
    }
)

# check_dup(result, "po_num")

In [0]:
result.write.mode(
    "overwrite"
).option(
    "partitionOverwriteMode", "dynamic"
).partitionBy(
    "image_date"
).parquet(
    f"{out_path}raw/"
)

result_pd = result.toPandas()
print(result_pd.shape)

In [0]:
final = result_pd.copy()

# Define binning rules
claim_count_bins = [-float('inf'), 2, 4, 6, 11, 16, 21, float('inf')]
claim_count_labels = ['a. 1', 'b. 2 - 3', 'c. 4 - 5', 'd. 6 - 10', 'e. 11 - 15', 'f. 16 - 20', 'g. >20']

claim_amount_bins = [-float('inf'), 5000.001, 7000, 10000, 15000, 20000, 30000, float('inf')]
claim_amount_labels = ['a. <=5M', 'b. 5 - 7M', 'c. 7 - 10M', 'd. 10 - 15M', 'e. 15 - 20M', 'f. 20 - 30M', 'g. >30M']

# Apply binning to appropriate columns
for col in final.columns:
    if col.endswith('_claims'):
        create_categorical(final, col, claim_count_bins, claim_count_labels)
    elif col.endswith('_claim_amount'):
        create_categorical(final, col, claim_amount_bins, claim_amount_labels)

tnps_amount_bins = [-float('inf'), 1282.1, 2564.1, 5128.2, 7692.3, 12820.5, float('inf')]
tnps_amount_labels = ['<$50', '$50 - $100', '$100 - $200', '$200 - $300', '$300 - $500', '>$500']

create_categorical(final, 'tnps_claim_amount', tnps_amount_bins, tnps_amount_labels)

In [0]:
from datetime import datetime

# Ensure 'frst_iss_dt' is in datetime format
final['frst_iss_dt'] = pd.to_datetime(final['frst_iss_dt'], errors='coerce')

# Calculate tenure in months
today = pd.to_datetime(datetime.today())
final['tenure_months'] = (today.year - final['frst_iss_dt'].dt.year) * 12 + (today.month - final['frst_iss_dt'].dt.month)

# Define bins and labels
tenure_bins = [-float('inf'), 6, 12, 36, 60, 84, 120, float('inf')]
tenure_labels = ['a. <=6M', 'b. 6 - 12M', 'c. 1 - 3yr', 'd. 3 - 5yr', 'e. 5 - 7yr', 'f. 7 - 10yr', 'g. >10yr']

# Create the 'client_tenure' column
final['client_tenure'] = pd.cut(final['tenure_months'], bins=tenure_bins, labels=tenure_labels, include_lowest=True)

In [0]:
def categorize_benefits(claim_benefits):
    if "/" in claim_benefits:
        if "Healthcare" in claim_benefits:
            if "Death" in claim_benefits:
                return "Healthcare > Death"
            elif "Disease" in claim_benefits:
                return "Healthcare > Disease"
            else:
                return "Healthcare > Other Major"
        elif "Medicash" in claim_benefits:
            if "Death" in claim_benefits:
                return "Medicash > Death"
            elif "Disease" in claim_benefits:
                return "Medicash > Disease"
            else:
                return "Medicash > Other Major"
        else:
            return "Only Major Claims"
    else:
        if "Healthcare" in claim_benefits:
            return "Healthcare"
        elif "Medicash" in claim_benefits:
            return "Medicash"
        else:
            return "Major (Disease/Death/Other)"

def categorize_rider_type(rider_type):
    if rider_type in ["MC Only", "HC Only", "MC & HC"]:
        return "MC/HC/Both Only"
    else:
        return rider_type

# Apply the functions to create a new columns
final['benefits_cat'] = final['claim_benefits'].apply(categorize_benefits)
final['rider_type_cat'] = final['rider_type'].apply(categorize_rider_type)

#display(final.head(5))

### 3_Analysis section

In [0]:
final.to_parquet(f"/dbfs{out_path}", partition_cols=['image_date'], engine='pyarrow', index=False)
print(final.shape)

In [0]:
final.to_csv(
    f"/dbfs{out_path}csv/claims_analysis_data.csv",
    index=False,
    header=True,
    encoding='utf-8-sig'
)